In [1]:
import numpy as np
import os
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


In [2]:
DATA_PATH = "/kaggle/input/project/GSE44861_series_matrix.txt"

def load_geo_series_matrix(path):
    data = []
    start = False

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("!series_matrix_table_begin"):
                start = True
                continue

            if line.startswith("!series_matrix_table_end"):
                break

            if start:
                if line.startswith("ID_REF") or line.startswith('"ID_REF"'):
                    continue

                parts = line.split("\t")
                values = [v.strip().strip('"') for v in parts[1:]]
                data.append(values)

    X = np.array(data, dtype=float)
    return X.T  # samples x features


In [3]:
class PreprocessPipeline:
    def __init__(self, use_pca=True, n_components=10):
        self.use_pca = use_pca
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components) if use_pca else None

    def fit(self, X):
        X_scaled = self.scaler.fit_transform(X)

        if self.use_pca:
            return self.pca.fit_transform(X_scaled)

        return X_scaled

    def transform(self, X):
        X_scaled = self.scaler.transform(X)

        if self.use_pca:
            return self.pca.transform(X_scaled)

        return X_scaled


In [4]:
X = load_geo_series_matrix(DATA_PATH)
print("Raw data shape:", X.shape)

prep = PreprocessPipeline(use_pca=True, n_components=10)
# TẠO THƯ MỤC TRƯỚC
os.makedirs("saved_model", exist_ok=True)
X_proc = prep.fit(X)
print("After preprocessing shape:", X_proc.shape)
np.save("saved_model/X_pca.npy", X_proc)
# LƯU preprocessor
os.makedirs("saved_model", exist_ok=True)
joblib.dump(prep, "saved_model/preprocessor.pkl")


Raw data shape: (111, 22277)
After preprocessing shape: (111, 10)


['saved_model/preprocessor.pkl']

In [5]:
# LOAD lại preprocessor
prep_loaded = joblib.load("saved_model/preprocessor.pkl")

X_new = X[:5]
X_new_proc = prep_loaded.transform(X_new)

print("New data shape:", X_new_proc.shape)


New data shape: (5, 10)
